# Day 2: Variational Monte Carlo for Fermionic Systems

Today we extend the VMC framework from Day 1 to study **interacting fermionic systems**.

We will work with the **t-V model** for spinless fermions on a 2D square lattice:

$$\hat{H} = -t \sum_{\langle i,j \rangle} (\hat{c}_i^\dagger \hat{c}_j + \hat{c}_j^\dagger \hat{c}_i) + V \sum_{\langle i,j \rangle} \hat{n}_i \hat{n}_j$$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh

torch.manual_seed(42)
np.random.seed(42)

## Problem 1: Jordan-Wigner Transformation

The Jordan-Wigner (JW) transformation maps fermionic operators to spin operators:

$$c_i^\dagger = \left(\prod_{j<i} -\sigma_j^z\right) \sigma_i^+, \qquad c_i = \left(\prod_{j<i} -\sigma_j^z\right) \sigma_i^-$$

We need to verify that these satisfy the **canonical anticommutation relations**:

$$\{c_i, c_j^\dagger\} = \delta_{ij}, \qquad \{c_i, c_j\} = \{c_i^\dagger, c_j^\dagger\} = 0$$


In [ ]:
# Problem 1: Jordan-Wigner transformation (rough guideline)

# Define local spin operators and tensor-product helpers.
# Then construct c_i and c_i^\dagger from the JW string and local raising/lowering operators.

I2  = np.eye(2, dtype=complex)
sz  = np.array([[1, 0], [0, -1]], dtype=complex)
sp  = np.array([[0, 1], [0,  0]], dtype=complex)
sm  = np.array([[0, 0], [1,  0]], dtype=complex)

def kron_op(op, site, L):
    """Place a single-site operator on the chosen site of an L-site chain."""
    # TODO: build the tensor product operator
    pass

def jw_c_dag(site, L):
    """Construct the fermionic creation operator using the JW transformation."""
    # TODO: combine the JW string with sigma^+ on the target site
    pass

def jw_c(site, L):
    """Construct the fermionic annihilation operator using the JW transformation."""
    # TODO: combine the JW string with sigma^- on the target site
    pass


In [ ]:
# Problem 1: numerical check of anticommutation relations

# TODO:
# 1. Choose a small system size L.
# 2. Build all c_i and c_i^\dagger operators.
# 3. Check the canonical anticommutation relations numerically.
# 4. Compare the result to the expected identity/zero matrices.

def anticommutator(A, B):
    """Return {A, B} = AB + BA."""
    # TODO
    pass



## Problem 2: The t-V Hamiltonian with Jordan-Wigner

We implement the t-V Hamiltonian for spinless fermions:

$$\hat{H} = -t \sum_{\langle i,j \rangle} (c_i^\dagger c_j + c_j^\dagger c_i) + V \sum_{\langle i,j \rangle} n_i n_j$$

For VMC we need Exact diagonalization (ED) to benchmark against

In [ ]:
# Problem 2: Hilbert-space basis for spinless fermions

def build_fock_basis(L, N):
    """Return all occupation-number configurations with N fermions on L sites."""
    # TODO: generate all binary configurations with exactly N occupied sites
    pass

def build_basis_dict(basis):
    """Map each basis configuration to its integer index in the basis list."""
    # TODO
    pass

def state_to_index(state, basis_dict):
    """Return the basis index of a given occupation configuration."""
    # TODO
    pass


In [ ]:
# Problem 2: t-V Hamiltonian with Jordan-Wigner signs
# The structure below is only an orientation; using it is optional!

def nearest_neighbor_bonds(Lx, Ly, pbc=True):
    """Return nearest-neighbor bonds for a square lattice with a chosen 1D ordering."""
    # TODO: define the lattice indexing and list all nearest-neighbor pairs
    pass

def fermionic_hop_sign(state, i, j):
    """Return the sign associated with hopping a fermion from j to i."""
    # TODO: compute the Jordan-Wigner parity between the two sites
    pass

def connected_elements_tv(state, bonds, t, V):
    """Return connected configurations and matrix elements for the t-V Hamiltonian."""
    # TODO:
    # - add diagonal density-density contributions
    # - add off-diagonal hopping contributions where allowed
    pass

def build_tv_hamiltonian(Lx, Ly, Nf, t, V, pbc=True):
    """Assemble the sparse t-V Hamiltonian in the fixed-particle-number Fock basis."""
    # TODO: build the basis, loop over states, and fill the sparse matrix
    pass

def groundstate_ed_tv(Lx, Ly, Nf, t, V, pbc=True):
    """Compute a small-system ED benchmark for the t-V model."""
    # TODO: diagonalize the sparse Hamiltonian and return the lowest states
    pass


## Problem 3: Slater Determinant Ansatz

The Slater determinant wavefunction is:

$$\psi(\sigma) = \det \tilde{A}(\sigma)$$

where $\tilde{A}(\sigma)$ is the $N \times N$ submatrix of $A$ (shape $L \times N$) obtained by selecting the rows corresponding to occupied orbitals.

The matrix $A$ contains all variational parameters.

In [ ]:
# Problem 3: Slater determinant ansatz

class SlaterDeterminant(nn.Module):
    """Single-determinant variational wavefunction for fixed particle number."""

    def __init__(self, n_sites, n_fermions):
        super().__init__()
        self.n_sites = n_sites
        self.n_fermions = n_fermions

        # TODO: initialize the trainable orbital matrix A

    def forward(self, sigma):
        """Evaluate psi(sigma) for a batch of occupation-number configurations."""
        # TODO:
        # 1. identify the occupied sites in each configuration
        # 2. slice the corresponding rows of A
        # 3. return the determinant of the resulting square matrix
        pass

# Optional sanity check:
# - build a small Fock basis
# - evaluate the model on a few configurations
# - inspect the output shape


In [ ]:
# Problem 3/4: VMC training for fermionic ansätze

def propose_fermion_hop(state):
    """Propose a particle-number-conserving update."""
    # TODO
    pass

def metropolis_sampler_fermions(model, n_samples, n_burn, basis_or_state_info):
    """Sample configurations from |psi_theta(sigma)|^2."""
    # TODO: adapt the Day-1 Metropolis sampler to fixed-particle-number configurations
    pass

def local_energy_tv(model, samples, hamiltonian_data):
    """Evaluate the local energy for the t-V model."""
    # TODO: use connected_elements_tv(...) and wavefunction ratios
    pass

def vmc_train_fermions(model, hamiltonian_data, n_iter=100, n_samples=300, lr=5e-3):
    """Train a fermionic variational state with the VMC loop from Day 1."""
    # TODO:
    # 1. sample configurations
    # 2. compute local energies
    # 3. estimate the energy gradient
    # 4. update parameters
    # 5. store the energy history
    pass

# TODO:
# Train the Slater determinant ansatz and compare its energy to ED.
# Optionally repeat the comparison with a simple FFNN ansatz.



## Problem 7: Hidden Fermion Determinant States (HFDS)

We now build the most expressive ansatz of the day. The key idea is to augment the physical system with $N_h$ hidden fermions and construct a Slater determinant on the enlarged system.

The wavefunction amplitude is:

$$\psi(\sigma) = \det M(\sigma), \quad M(\sigma) = \begin{pmatrix} \phi_v & \phi_h \\ \chi_v(\sigma) & \chi_h(\sigma) \end{pmatrix}$$

- $\phi_v, \phi_h$: trainable parameter blocks (fixed, configuration-independent)
- $\chi_v(\sigma), \chi_h(\sigma)$: neural-network-generated rows (configuration-dependent)

The matrix $M(\sigma)$ has size $(N + N_h) \times (N + N_h)$.

In [ ]:
# Problem 7: Hidden Fermion Determinant State (HFDS)

class HiddenFermionDeterminantState(nn.Module):
    """Determinant ansatz with additional hidden fermionic degrees of freedom."""

    def __init__(self, n_sites, n_fermions, n_hidden, hidden_dims=(32,)):
        super().__init__()
        self.n_sites = n_sites
        self.n_fermions = n_fermions
        self.n_hidden = n_hidden
        self.n_total = n_fermions + n_hidden

        # TODO:
        # - define the configuration-independent determinant blocks
        # - define a neural network that outputs the configuration-dependent hidden block

    def forward(self, sigma):
        """Evaluate the HFDS wavefunction amplitude for a batch of configurations."""
        # TODO:
        # 1. slice the physical rows according to occupied sites
        # 2. generate the hidden block from the neural network
        # 3. assemble the full determinant matrix M(sigma)
        # 4. return det M(sigma)
        pass

# TODO:
# Train the HFDS ansatz with the same VMC machinery.
# Compare to ED and study the dependence on n_hidden / network depth.
# Optionally scan t/V and look for signatures of charge order.


## Problem 8: Comparison to Exact Results

We optimize the HFDS for different levels of model complexity, and then compare to the exact results. 


In [ ]:
def run_scaling_study(param_list):
    """
    Study how the variational energy depends on model complexity.

    param_list could include:
      - number of hidden fermions
      - number of hidden layers
    """

    results = []

    for params in param_list:

        # TODO: build model with given parameters

        # TODO: run VMC optimization

        # TODO: store final energy (and optionally other metrics)

        pass

    return results


# TODO:
# - define a range of model parameters
# - run scaling study
# - plot energy vs model complexity and compare to ED

## Problem 9: Locating the Phase Transition

We optimize the NQS for different values of $t/V$. 


In [ ]:
# Problem 9: scan over t/V and locate phase transition

def scan_phase_diagram(t_values, V, system_params):
    """
    Run VMC for different values of t/V and track observables.

    Goal: identify signatures of the phase transition.
    """

    observables = []

    for t in t_values:

        # TODO: build Hamiltonian with current t, V

        # TODO: optimize NQS (VMC)

        # TODO: measure relevant observables (e.g. density correlations)

        # TODO: store results

        pass

    return observables


# TODO:
# - choose a parameter range for t/V
# - run scan
# - plot observables vs t/V
# - identify transition point